# Fire / Smoke / Normal detector (model CA)

**Model CA:** resize to 256 (area) → noise σ (Immerkær). If σ > 2.56: median 3×3 + CLAHE on L (clip 2.0, tile 8). Otherwise unchanged. Then MobileViT-XS trained with Gaussian noise.

**Google Drive folder `My Drive/firesmoke/`:**
- `firesmoke_CA.pt`: the model (uploaded once)
- `firesmoke_code.zip`: backup copy of the code
- `input/`: put photos here (when `INPUT_MODE = "drive"`)
- `outputs/`: every run is saved here

Run the cells from top to bottom: **Runtime → Run all**.

## 1. GitHub info and settings

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME   = "dip_fire_smoke"
CODE_SOURCE = "github"
INPUT_MODE  = "upload"
SHOW_N      = 5
DRIVE_DIR   = "/content/drive/MyDrive/firesmoke"

`CODE_SOURCE`: `"github"` or `"drive"` (the other one is used if the first fails).  
`INPUT_MODE`: `"upload"` (pick photos from this computer) or `"drive"` (use `My Drive/firesmoke/input/`).  
`SHOW_N`: how many photos to show in steps 9 and 10.

## 2. Connect Google Drive

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess, datetime
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
for sub in ["input", "outputs"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
print("Drive folder:", DRIVE_DIR)
print("Contents:", sorted(os.listdir(DRIVE_DIR)))

## 3. Get the code

In [ ]:
CODE_DIR = "/content/code"
shutil.rmtree(CODE_DIR, ignore_errors=True)
shutil.rmtree("/content/code_zip", ignore_errors=True)

def from_github():
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
    ok = subprocess.run(["git", "clone", "-q", "--depth", "1", url, CODE_DIR]).returncode == 0
    return CODE_DIR if ok and os.path.isfile(f"{CODE_DIR}/firesmoke/__init__.py") else None

def from_drive():
    z = f"{DRIVE_DIR}/firesmoke_code.zip"
    if not os.path.isfile(z):
        return None
    zipfile.ZipFile(z).extractall("/content/code_zip")
    found = glob.glob("/content/code_zip/**/firesmoke/__init__.py", recursive=True)
    return str(Path(found[0]).parent.parent) if found else None

order = [from_github, from_drive] if CODE_SOURCE == "github" else [from_drive, from_github]
code_root = order[0]() or order[1]()
if not code_root:
    raise SystemExit("No code found. Check GITHUB_USER, or put firesmoke_code.zip in My Drive/firesmoke/.")
print("Code from:", code_root)

## 4. Install

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
[sys.modules.pop(m) for m in list(sys.modules) if m.startswith("firesmoke")]
sys.path.insert(0, code_root)

import torch, firesmoke
from firesmoke import FireSmokeDetector, Result

NEW_CODE = hasattr(Result, "show_compare")
print("firesmoke from:", firesmoke.__file__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU (fine, just slower)")
print("Input vs output comparison:", "yes" if NEW_CODE else "NO, old code: push the new firesmoke folder to GitHub or update firesmoke_code.zip")

## 5. Model weights

In [ ]:
DRIVE_PT = f"{DRIVE_DIR}/firesmoke_CA.pt"
LOCAL_PT = "/content/firesmoke_CA.pt"

if not os.path.isfile(DRIVE_PT):
    print("firesmoke_CA.pt is not in Drive yet. Upload it now (only this once).")
    os.chdir("/content")
    up = files.upload()
    name = next(iter(up))
    shutil.move(f"/content/{name}", DRIVE_PT)
shutil.copy(DRIVE_PT, LOCAL_PT)
det = FireSmokeDetector(LOCAL_PT)
print("Model:", det.meta.get("model_name"), "|", round(os.path.getsize(LOCAL_PT) / 1e6, 1), "MB")

## 6. Input images

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PHOTOS = "/content/photos"
shutil.rmtree(PHOTOS, ignore_errors=True)
os.makedirs(PHOTOS)

if INPUT_MODE == "drive":
    src = [p for p in Path(f"{DRIVE_DIR}/input").rglob("*") if p.suffix.lower() in EXT]
    [shutil.copy(p, f"{PHOTOS}/{p.parent.name}_{p.name}" if p.parent.name != "input" else f"{PHOTOS}/{p.name}") for p in src]
else:
    os.chdir(PHOTOS)
    files.upload()
    os.chdir("/content")

photos = sorted(p for p in Path(PHOTOS).iterdir() if p.suffix.lower() in EXT)
if not photos:
    raise SystemExit("No photos found. Upload some, or put them in My Drive/firesmoke/input/.")
RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUT_DIR = f"/content/outputs/{RUN_ID}"
os.makedirs(OUT_DIR, exist_ok=True)
print(RUN_ID, "|", len(photos), "photos:", [p.name for p in photos])

## 7. Run the model

In [ ]:
results, skipped = [], []
for p in photos:
    try:
        r = det.predict(p)
    except Exception as e:
        skipped.append(p.name)
        print(f"{p.name}: SKIPPED ({e})")
        continue
    r.save(OUT_DIR)
    results.append(r)
    extra = f"  PSNR {r.change['psnr_db']:.2f} dB  SSIM {r.change['ssim']:.3f}" if NEW_CODE else ""
    print(f"{p.name}: {r.label.upper()} {r.confidence * 100:.1f}%  (sigma {r.sigma:.2f}, {r.decision}, {r.filter}){extra}")
print(f"\n{len(results)} done, {len(skipped)} skipped")

## 8. Results table

In [ ]:
import pandas as pd

table = pd.DataFrame([{
    "photo": Path(r.image).name,
    "label": r.label,
    "confidence": round(r.confidence, 3),
    **{f"p_{k}": round(v, 3) for k, v in r.probs.items()},
    "sigma": round(r.sigma, 2),
    "decision": r.decision,
    "filter": r.filter,
    **(r.change if NEW_CODE else {}),
} for r in results])
table.to_csv(f"{OUT_DIR}/results_table.csv", index=False)
print(table["label"].value_counts().to_string())
table

## 9. Input vs output (how much the DIP step changed each photo)

In [ ]:
if NEW_CODE:
    [r.show_compare() for r in results[:SHOW_N]]
else:
    print("Needs the new firesmoke code (see step 4).")

## 10. All DIP steps and the explanation

In [ ]:
for r in results[:SHOW_N]:
    r.explain()
    r.show_steps()
    print()

## 11. Save results to Google Drive and download

In [ ]:
drive_out = f"{DRIVE_DIR}/outputs/{RUN_ID}"
shutil.copytree(OUT_DIR, drive_out)
zip_path = shutil.make_archive(f"/content/{RUN_ID}", "zip", "/content/outputs", RUN_ID)
print("Saved in Drive:", drive_out)
files.download(zip_path)